# Simulation 
### Model construction
The model in this notebook is similar to the one presented in `sir-teaching.ipynb`, but adapted so that we can implement some interventions.
We start off with the same installs and imports, which you can ignore.

In [ ]:
%pip install summerepi2==1.3.6

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
pd.options.plotting.backend = "plotly"

from summer2 import CompartmentalModel
from summer2.parameters import Parameter

### Building the model
This time, let's build a slightly more complicated model so that we can distinguish two phases
of the infectious period.
We'll add `_asympt` and `_sympt` to indicate that we're considering that the
first compartment is going to be considered asymptomatic, 
whereas the second will be considered symptomatic.
We'll also include a vaccinated compartment, which we'll come back to below.

In [ ]:
total_population = 7e6
infectious_seed = 1.0
run_period = [0.0, 40.0]
model_comps = ["vaccinated", "susceptible", "infectious_asympt", "infectious_sympt", "recovered"]
infect_comps = ["infectious_asympt", "infectious_sympt"]
sir_model = CompartmentalModel(times=run_period, compartments=model_comps, infectious_compartments=infect_comps, timestep=0.2)

## Interventions
### Face masks
Let's implement population-wide use of face masks into our simple model.
Rather than the infection rate being determined by the contact rate
parameter alone, we'll adjust this based on the proportion of people
wearing the masks ("coverage") and the efficacy of wearing the mask
on preventing transmission in those who are wearing them.

In [ ]:
infection_rate = Parameter("contact_rate") * (1.0 - Parameter("face_mask_coverage") * Parameter("face_mask_efficacy"))
sir_model.add_infection_frequency_flow(name="infection", contact_rate=infection_rate, source="susceptible", dest="infectious_asympt")

### Vaccination
Let's allow that vaccination can reduce the rate at which people are infected.
We included a vaccinated compartment earlier,
so we'll have to apply an infection process to this compartment too,
but adjust the rate at which people from this compartment are infected
according to the efficacy of the vaccine being used.
We'll only see an effect from this if we start some of the 
population off in the vaccinated compartment, of course.

In [ ]:
vacc_infection_rate = infection_rate * (1.0 - Parameter("vacc_efficacy"))
sir_model.add_infection_frequency_flow(name="infection_vacc", contact_rate=vacc_infection_rate, source="vaccinated", dest="infectious_asympt")

### Case isolation
We'll now look at incorporating the effect of case isolation for infected people with symptoms.
Before we can start on this, we need to make an adjustment to the way we are simulating recovery.
Because people now need to progress through two compartments instead of one to recover,
to retain similar behaviour to what we had with the simple SIR model,
we need to double this rate of progression.
(By doing this, the average time infectious will still be the reciprocal
of the rate of recovery.)
Now that we've done that, we can add an additional rate at which people with symptoms
effectively remove themselves from the infectious population,
which we can refer to as case isolation.
Because this only applies to the compartment with symptoms (i.e. the second
half of the infectious period),
we can add this on to the rate of transition from `infectious_sympt` to `recovered`.

In [ ]:
progression_rate = Parameter("recovery_rate") * 2.0
sir_model.add_transition_flow(name="progression", fractional_rate=progression_rate, source="infectious_asympt", dest="infectious_sympt")
resolve_sympt_rate = progression_rate + Parameter("isolation_rate")
sir_model.add_transition_flow(name="recovery", fractional_rate=resolve_sympt_rate, source="infectious_sympt", dest="recovered")

### Preparing the model

In [ ]:
suscept_pop = total_population - infectious_seed
start_pop = {
    "susceptible": suscept_pop * (1.0 - Parameter("vacc_coverage")),
    "vaccinated": suscept_pop * Parameter("vacc_coverage"),
    "infectious_asympt": infectious_seed,
}
sir_model.set_initial_population(start_pop)

### Running the model with interventions
Now we have a model that can run three different interventions,
and any combination of those three interventions together.
This cell is for you to experiment with different values 
for the five different intervention-related input parameters.
See how they affect the epidemic and whether the results are
consistent with what you expected.

Note that if you want to change any of the structures of the model above,
you will probably need to re-run all the preceding cells,
but if you're happy with the structure, you should be able 
to adjust the parameters and re-run in this cell alone.

In [ ]:
parameters = {
    "contact_rate": 1.5,
    "recovery_rate": 0.2,
    "face_mask_coverage": 0.0,
    "face_mask_efficacy": 0.0,
    "isolation_rate": 0.0,
    "vacc_efficacy": 0.0,
    "vacc_coverage": 0.0,
}
sir_model.run(parameters)
outputs = sir_model.get_outputs_df()

# Rather than using the outputs directly, we can collapse the infectious and the never infected comparment values together
broad_state_outputs = pd.DataFrame(index=outputs.index)
broad_state_outputs["infectious"] = outputs[infect_comps].sum(axis=1)
broad_state_outputs["never_infected"] = outputs[["susceptible", "vaccinated"]].sum(axis=1)
broad_state_outputs["recovered"] = outputs["recovered"]
broad_state_outputs.plot(labels={"index": "time", "value": "number of people"})